# 07. Ablations, representation checks, and the paper evidence ledger

We will inspect representations without fitting on held-out data, define a manageable ablation plan, and distinguish completed checks from evidence still needed for a paper.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

In [ ]:
from aaai_gait.workflow import prepare_corpus
x, meta, provenance = prepare_corpus(settings)
print('Window tensor:', x.shape, '[windows, frames, selected joints, XY]')
display(meta.groupby(['split','label']).agg(windows=('sequence_id','size'), sequences=('sequence_id','nunique'), sources=('source_id','nunique')))

## 1. Fit visualization on training data

A two-dimensional projection can help find outliers and source clusters, but it cannot prove meaningful class separation. PCA is fitted on training embeddings only. Validation embeddings are transformed with that fitted map. Test plots should be produced only after the analysis has been locked.

In [ ]:
from aaai_gait.workflow import normal_anchor
from aaai_gait.training import embed_windows, representation_diagnostics
from sklearn.decomposition import PCA
anchor,_=normal_anchor(x,meta,provenance,settings)
z=embed_windows(anchor,x,batch_size=settings.batch_size)
train=meta.split.eq('train').to_numpy();val=meta.split.eq('val').to_numpy()
pca=PCA(n_components=2).fit(z[train]);points=pca.transform(z[val])
fig,ax=plt.subplots(figsize=(7,4))
for label in LABELS:
    keep=meta.loc[val,'label'].to_numpy()==label
    ax.scatter(points[keep,0],points[keep,1],label=label)
ax.legend(fontsize=8);ax.set_title('Synthetic validation projection' if settings.demo else 'Validation features in training-fitted PCA');plt.show()
display(representation_diagnostics(z[train]))

## 2. Predefine ablations before reading their test scores

|Comparison|Fixed elements|Question|
|---|---|---|
|Uniform JEPA with and without VICReg|Sources, seed, steps, joint whitelist|Does the encoder retain useful variance?|
|Deterministic positions versus StoP-inspired noise|Sources, seeds, compute budget|Does positional noise improve held-out features?|
|Normal anchor versus progressive adaptation|Readout labels and validation procedure|Does adaptation help condition recognition?|
|Progressive versus pooled training|Total optimizer steps and label exposure|Does stage order matter beyond compute?|
|Masked completion versus suffix forecasting|Temporal scale and normal sources|Does future prediction add anomaly information?|
|Untrained encoder and mean pose controls|Same classifiers and splits|Does learned temporal structure add value?|

The main pipeline implements the normal/progressive comparison and simple controls. The optional SPE training call below implements a position-noise candidate. Pooled, reversed-order, and compute-matched schedules remain additional experiments to run and report, not completed results.

In [ ]:
# Optional locally configured ablation. Leave off for the short walkthrough.
if os.getenv('AAAI_RUN_SPE_ABLATION','0')=='1':
    spe_anchor,spe_history=normal_anchor(x,meta,provenance,settings,spe=True)
    print(pd.DataFrame(spe_history).tail(1))
else:
    print('SPE ablation not run. Enable AAAI_RUN_SPE_ABLATION=1 in the root .env.')

## 3. Test robustness on development data first

Pose jitter is a technical perturbation, not a disease simulation. The example measures how a small coordinate perturbation changes validation embeddings. Camera view, lower frame rate, missing joints, and person identity require separate controlled audits. The validity gate rejects windows with too little observed motion. In retained windows, missing tokens are excluded from attention and target loss. The pose-availability control combines per-joint and per-time-patch validity with training-fitted camera-view indicators. It checks whether recording quality and view can predict labels.

In [ ]:
rng=np.random.default_rng(23)
perturbed=x[val]+rng.normal(0,.01,x[val].shape).astype(np.float32)
original=embed_windows(anchor,x[val],batch_size=settings.batch_size)
changed=embed_windows(anchor,perturbed,batch_size=settings.batch_size)
print('Mean validation embedding displacement after 0.01 torso-unit noise:',np.linalg.norm(changed-original,axis=1).mean())

## 4. Assemble an honest evidence ledger

A notebook that executes demonstrates an implementation. It does not demonstrate anomaly-detection performance. Save separately: the full annotation inventory, downloaded media inventory, pose exclusions, final retained cohort, split registry, checkpoints, validation selection, per-sequence predictions, and statistical summaries.

|Evidence|Meaning|
|---|---|
|Synthetic notebook execution|Software paths run with explicit toy data|
|Model invariant tests|Selected leakage, gradient, and masking properties hold|
|Real-media extraction pilot|The ambient integration can process some actual annotations|
|Full GAVD evaluation|Required before claiming within-dataset performance|
|Independent cohort evaluation|Required for stronger generalization claims|

The paper should lead with normal-only anomaly detection, then present condition classification as supporting analysis. Myopathic gait broadens the cohort beyond exclusively neurological disease. Source labels, disease severity, participant identities, and filming bias limit interpretation. The [paper plan](docs/paper_plan.md) contains a claim-to-evidence map and the verified submission rules.

In [ ]:
from aaai_gait.workflow import write_json
import importlib.metadata as im
versions={name:im.version(name) for name in ['torch','numpy','pandas','scikit-learn','mediapipe','opencv-python','python-dotenv']}
write_json(provenance['run_dir']/'software_versions.json',versions)
display(pd.Series(versions))
print('Use the exact run directory in the experiment ledger:',provenance['run_dir'])

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.